# 层和块

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand(2,20)
X.shape # batch_size=2, in_features=20

torch.Size([2, 20])

In [3]:
net(X) # return 10 logits

tensor([[-0.0600,  0.1918, -0.0498, -0.2185, -0.1146, -0.0321, -0.0198,  0.1143,
         -0.0233, -0.1131],
        [-0.2065,  0.1771, -0.0464, -0.3060, -0.2447, -0.0031, -0.0936,  0.0899,
         -0.0827, -0.1510]], grad_fn=<AddmmBackward0>)

In [4]:
# 自定义网络层、完整模型都必须继承自 nn.Module
class MLP(nn.Module):
    '''nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))'''
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [5]:
net = MLP()
net(X)

tensor([[-0.0904, -0.2138, -0.1036, -0.0736,  0.0271, -0.0369, -0.0019,  0.0486,
          0.0866, -0.1548],
        [-0.1199, -0.1072, -0.0063,  0.1393, -0.0308, -0.1423, -0.1629,  0.1156,
          0.0171, -0.2286]], grad_fn=<AddmmBackward0>)

# 参数管理

In [6]:
import torch
from torch import nn

In [7]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand(2,20)
net(X)

tensor([[ 0.0399, -0.0425, -0.0896,  0.0416, -0.0298, -0.1156, -0.1416, -0.0755,
         -0.1538,  0.1287],
        [ 0.0487, -0.1321, -0.0718,  0.1377, -0.1135, -0.1853, -0.2408,  0.0609,
         -0.1643,  0.1696]], grad_fn=<AddmmBackward0>)

In [8]:
# .state_dict()返回一个有序字典OrderedDict，包含所有可学习参数
print(net[2].state_dict())

OrderedDict({'weight': tensor([[ 0.0341,  0.0125, -0.0354,  ..., -0.0120, -0.0401,  0.0548],
        [-0.0475, -0.0348, -0.0598,  ..., -0.0106, -0.0224, -0.0141],
        [ 0.0534, -0.0139, -0.0068,  ...,  0.0054,  0.0417,  0.0045],
        ...,
        [ 0.0448, -0.0259,  0.0256,  ..., -0.0268, -0.0004, -0.0596],
        [-0.0613,  0.0403, -0.0360,  ..., -0.0585, -0.0190,  0.0364],
        [ 0.0585, -0.0064, -0.0539,  ...,  0.0511,  0.0041, -0.0421]]), 'bias': tensor([ 0.0284, -0.0233, -0.0151,  0.0180,  0.0093,  0.0265, -0.0593,  0.0430,
        -0.0304,  0.0195])})


In [9]:
# .state_dict返回方法的内存地址
print(net[2].state_dict)

<bound method Module.state_dict of Linear(in_features=256, out_features=10, bias=True)>


In [10]:
# net[2]层偏置bias的数据类型
print(type(net[2].bias))

<class 'torch.nn.parameter.Parameter'>


In [11]:
# net[2]层bias的参数内容
print(net[2].bias)

Parameter containing:
tensor([ 0.0284, -0.0233, -0.0151,  0.0180,  0.0093,  0.0265, -0.0593,  0.0430,
        -0.0304,  0.0195], requires_grad=True)


In [12]:
# 打印bias的数据张量，不含梯度信息
print(net[2].bias.data)

tensor([ 0.0284, -0.0233, -0.0151,  0.0180,  0.0093,  0.0265, -0.0593,  0.0430,
        -0.0304,  0.0195])


In [13]:
# 前向传播后未执行反向传播，梯度尚未计算
net[2].weight.grad == None

True

In [14]:
# *解包列表，使每个元组独立打印
# 打印第一层net[0]所有参数的名称和形状
print(*[(name, param.shape) for name, param in net[0].named_parameters()])

('weight', torch.Size([256, 20])) ('bias', torch.Size([256]))


In [15]:
# 打印整个网络所有层的参数名称和形状，跳过无参数的ReLU层
print(*[(name, param.shape) for name, param in net.named_parameters()])

('0.weight', torch.Size([256, 20])) ('0.bias', torch.Size([256])) ('2.weight', torch.Size([10, 256])) ('2.bias', torch.Size([10]))


In [16]:
# 访问整个网络参数字典中第三层偏置'2.bias'的数据
# ['2.bias']获取第三层偏置
# .data返回其张量值
net.state_dict()['2.bias'].data

tensor([ 0.0284, -0.0233, -0.0151,  0.0180,  0.0093,  0.0265, -0.0593,  0.0430,
        -0.0304,  0.0195])

In [17]:
def block1():
    '''
    包含3层的序列模块

    线性层：输入20维 → 输出256维
    ReLU激活函数
    线性层：输入256维 → 输出20维
    '''
    return nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 20))

def block2():
    '''
    4个相同的瓶颈结构串联

    添加4个独立的 block1模块,每个模块被命名为 block0, block1, block2, block3
    '''
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}', block1())
    return net

# 包含4个block1的block2模块 + 线性层：20维 → 2维输出
rgnet = nn.Sequential(block2(),nn.Linear(20,2))
rgnet(X)

tensor([[0.0290, 0.0413],
        [0.0289, 0.0413]], grad_fn=<AddmmBackward0>)

In [18]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=20, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=20, bias=True)
    )
    (block1): Sequential(
      (0): Linear(in_features=20, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=20, bias=True)
    )
    (block2): Sequential(
      (0): Linear(in_features=20, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=20, bias=True)
    )
    (block3): Sequential(
      (0): Linear(in_features=20, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=20, bias=True)
    )
  )
  (1): Linear(in_features=20, out_features=2, bias=True)
)


In [19]:
def init_normal(m):
    '''
    参数初始化
    线性层:正态分布初始化权重, 均值=0, 标准差=0.01; 偏置初始化为全零
    '''
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

# ​​.apply方法
# 递归遍历网络的所有子模块,对每个模块应用init_normal()函数
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 1.6377e-03, -1.1035e-02,  1.2752e-02, -1.0655e-02, -1.8037e-03,
         -9.5050e-03, -3.4905e-03,  6.7756e-05, -7.1624e-03,  1.7143e-03,
          8.1133e-03,  1.0422e-02, -8.0876e-03,  5.4831e-04, -2.8194e-02,
         -2.2328e-02,  2.0777e-02,  1.0885e-02, -6.1353e-03,  1.2549e-03]),
 tensor(0.))

In [20]:
def init_constant(m):
    '''初始化参数为1，不常用'''
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1.]),
 tensor(0.))

In [21]:
def xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

net.apply(xavier)
net[0].weight.data[0]

tensor([-0.1067, -0.1076,  0.0099, -0.1318, -0.0798,  0.1446,  0.0423,  0.0393,
        -0.1200, -0.0865, -0.1231, -0.1431, -0.0170, -0.0892, -0.0170, -0.0567,
         0.0936, -0.1387,  0.1163,  0.0227])

In [22]:
# 参数绑定
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), shared, nn.ReLU(), shared, nn.ReLU(), nn.Linear(8,1))
# 2和4层参数相等

X = torch.rand(1,4)
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


# 自定义层

In [23]:
import torch
from torch import nn
from torch.nn import functional as F

In [24]:
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        '''零均值化,即新数据的均值为0'''
        return X - X.mean()

layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [25]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

Y = net(torch.rand(4, 8))
Y.mean()

tensor(-4.1910e-09, grad_fn=<MeanBackward0>)

In [26]:
# 带参数的图层
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        '''
        in_units：输入特征维度
        units：输出特征维度，神经元数量
        '''
        # super()调用父类 nn.Module的方法和属性​​，确保参数注册和设备管理被正确初始化
        super().__init__()
        # .Parameter()将张量标记为可学习参数，会被优化器自动更新
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight) + self.bias
        return F.relu(linear)

dense = MyLinear(5,3)
dense.weight

Parameter containing:
tensor([[-0.3146, -1.0186,  1.5396],
        [-0.5503,  0.3423, -1.6727],
        [ 1.0901,  0.7959, -0.4691],
        [ 0.0209,  0.8215,  0.5435],
        [ 1.2044,  0.7566, -1.2236]], requires_grad=True)

In [27]:
# 使用自定义层直接执行正向传播计算
dense(torch.rand(2, 5)) # 2个样本，每个样本5个特征

tensor([[1.1312, 0.0000, 0.0000],
        [0.1724, 0.0000, 0.0000]], grad_fn=<ReluBackward0>)

In [28]:
# 使用自定义层构建模型
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.],
        [0.]], grad_fn=<ReluBackward0>)

# 读写文件

In [29]:
import torch
from torch import nn
from torch.nn import functional as F

In [30]:
x = torch.arange(4)
torch.save(x, 'x-file')

In [31]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

In [32]:
# 存储一个张量列表，然后把它们读回内存
y = torch.zeros(4)
torch.save([x, y], 'x-file')
x2, y2 = torch.load('x-file')
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

In [33]:
# 写入或读入从字符串映射到张量的字典
mydict = {'x':x, 'y':y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

In [34]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

In [35]:
torch.save(net.state_dict(), 'mlp.params')

In [36]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [37]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])